# Phase 4: Regime Detection via Unsupervised Machine Learning

In this notebook, we classify historical months into four distinct regimes using unsupervised learning.

In [1]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_master_dataset, setup_plotting_style
from src.regime_detection import (
    engineer_features,
    perform_pca,
    detect_regimes,
    plot_regime_validation,
    plot_regime_sector_distributions
)

setup_plotting_style()
df = load_master_dataset()
print(f"Master dataset loaded. Shape: {df.shape}")

Master dataset loaded. Shape: (209, 35)


### 1. Feature Engineering

Build MoM, YoY, z-scores, and momentum factors for macro indicators.

In [2]:
features_df = engineer_features(df)
print(f"Features engineered. Shape: {features_df.shape}")
features_df.head()

  🔧 Engineering macro features...
Features engineered. Shape: (70, 44)


/Users/neil/Desktop/Macro/src/regime_detection.py:45: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  features[f"{macro}_MoM"] = df[macro].pct_change()
/Users/neil/Desktop/Macro/src/regime_detection.py:47: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  features[f"{macro}_YoY"] = df[macro].pct_change(12)
/Users/neil/Desktop/Macro/src/regime_detection.py:45: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  f

,US_Fed_Funds_Rate_MoM,US_Fed_Funds_Rate_YoY,US_Fed_Funds_Rate_ZScore_6M,US_Fed_Funds_Rate_Mom_3M,US_CPI_MoM,US_CPI_YoY,US_CPI_ZScore_6M,US_CPI_Mom_3M,US_10Y_Yield_MoM,US_10Y_Yield_YoY,...,USD_INR_ZScore_6M,USD_INR_Mom_3M,DXY_MoM,DXY_YoY,DXY_ZScore_6M,DXY_Mom_3M,Global_VIX_MoM,Global_VIX_YoY,Global_VIX_ZScore_6M,Global_VIX_Mom_3M
Date,,,,,,,,,,,,,,,,,,,,,
2010-12-31,-0.052632,0.500000,-1.290994,-0.01,0.004017,0.014378,1.517949,2.197,0.191064,-0.083312,...,-0.681282,0.120003,-0.026724,0.015027,-0.516524,0.309998,-0.245964,-0.181273,-1.716871,-5.950001
2011-01-31,-0.055556,0.545455,-1.792843,-0.02,0.003243,0.017008,1.405646,2.152,0.031326,-0.090850,...,0.464902,1.418003,-0.016323,-0.021646,-0.791529,0.470001,0.100282,-0.206742,-0.797729,-1.670000
2011-02-28,-0.058824,0.230769,-1.581139,-0.03,0.003214,0.021249,1.337683,2.308,0.053717,-0.031085,...,0.194145,-0.549999,-0.010934,-0.043181,-1.010635,-4.309998,-0.060420,-0.058974,-0.907587,-5.190001
2011-03-31,-0.125000,-0.125000,-1.631638,-0.04,0.005174,0.026192,1.462424,2.574,-0.045289,-0.083985,...,-0.842675,-0.200001,-0.013396,-0.064265,-1.136091,-3.169998,-0.033243,0.008528,-0.845018,-0.010000
2011-04-30,-0.285714,-0.500000,-1.735055,-0.07,0.004694,0.030772,1.431821,2.906,0.011906,-0.101855,...,-1.260317,-1.538002,-0.038624,-0.109198,-1.541375,-4.809998,-0.168546,-0.331066,-1.337973,-4.780001


### 2. PCA Dimensionality Reduction

Standardize features and run PCA to extract components explaining at least 85% of variance.

In [3]:
pca_df = perform_pca(features_df)
print(f"PCA complete. Components: {pca_df.shape[1]}")
pca_df.head()

  📉 Applying PCA for dimensionality reduction...
    ➡️ Using 13 components to explain 86.3% of variance
  📊 Saved chart: data/processed/charts/regime/pca_scree_plot.png
PCA complete. Components: 13


,0,1,2,3,4,5,6,7,8,9,10,11,12
Date,,,,,,,,,,,,,
2010-12-31,3.629560,-0.171983,-0.674016,-3.985209,-1.042546,-3.201555,0.411756,-1.454417,-0.013589,3.057734,0.802467,1.387345,-2.297083
2011-01-31,1.459974,-0.143549,2.521316,-1.907642,-0.800057,-1.263823,-0.242582,0.108197,1.168505,2.656953,-1.016767,-0.444452,-0.453639
2011-02-28,2.184072,0.802913,1.819599,-2.492317,-1.491804,-1.464885,0.064322,-0.087297,-0.138072,1.548625,-1.135598,-0.437898,0.021032
2011-03-31,1.798659,1.800744,1.516388,-2.232592,-1.395736,-0.817557,-0.470558,0.737164,-1.072263,-0.012696,-0.767388,-0.294872,-0.361189
2011-04-30,3.355087,2.528068,0.393814,-2.234062,-2.069765,-0.273430,-1.103627,0.999263,0.207840,-0.463294,-0.698392,-0.571514,-1.213740


### 3. K-Means Economic Regime Clustering

Apply K-Means ($k=4$) to cluster PCA data, mapping each cluster to Expansion, Peak, Contraction, or Recovery.

In [4]:
labels_df = detect_regimes(pca_df, df)
print("Regime distribution counts:")
print(labels_df["Regime"].value_counts())
labels_df.head()

  🤖 Clustering via K-Means (k=4)...
  💾 Saved regime labels: regime_labels.csv
Regime distribution counts:
Regime
Expansion      31
Peak           25
Recovery       12
Contraction     2
Name: count, dtype: int64


,Cluster,Regime
Date,,
2010-12-31,0,Peak
2011-01-31,0,Peak
2011-02-28,0,Peak
2011-03-31,0,Peak
2011-04-30,3,Expansion


### 4. Validation & Visualization

Overlay regime periods on a Nifty 50 historical timeline, and plot returns distributions per regime.

In [5]:
plot_regime_validation(labels_df, df)
plot_regime_sector_distributions(labels_df, df)

  📊 Plotting regime validation...
  📊 Saved chart: data/processed/charts/regime/regime_timeline.png
  📊 Plotting sector return distributions by regime...
  📊 Saved chart: data/processed/charts/regime/regime_sector_distributions.png
